# Hyperparameter Tuning

## Data Preparation

In [1]:
from pathlib import Path
import json
import sys

import mlflow
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory
    if (working_directory / "app").exists()
    else working_directory.parent
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.ml.preprocessing import build_preprocessor, prepare_features_and_target

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "fraud_detection.csv"
MLFLOW_DIRECTORY = PROJECT_ROOT / "mlruns"
MLFLOW_TRACKING_URI = f"sqlite:///{(MLFLOW_DIRECTORY / 'mlflow.db').resolve().as_posix()}"
RANDOM_STATE = 42
EXPERIMENT_NAME = "fraud_detection_model_tuning"

In [2]:
raw_data = pd.read_csv(DATA_PATH)
X, y = prepare_features_and_target(raw_data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Raw rows: {len(raw_data):,}")
print(f"Rows after exact deduplication: {len(X):,}")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
display(pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "train_percentage": y_train.value_counts(normalize=True).sort_index().mul(100),
    "test_count": y_test.value_counts().sort_index(),
    "test_percentage": y_test.value_counts(normalize=True).sort_index().mul(100),
}).rename(index={0: "Legitimate (0)", 1: "Fraudulent (1)"}))

Raw rows: 51,000
Rows after exact deduplication: 50,119
Training rows: 40,095; test rows: 10,024


,train_count,train_percentage,test_count,test_percentage
Fraudulent,,,,
Legitimate (0),38121,95.076693,9531,95.081804
Fraudulent (1),1974,4.923307,493,4.918196


In [3]:
MLFLOW_DIRECTORY.mkdir(exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

def make_random_forest_pipeline(
    classifier_params: dict | None = None,
    forest_n_jobs: int = 2,
) -> Pipeline:
    params = {
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": forest_n_jobs,
    }
    if classifier_params:
        params.update(classifier_params)
    return Pipeline(steps=[
        ("preprocessing", build_preprocessor(X_train)),
        ("classifier", RandomForestClassifier(**params)),
    ])

def evaluate_model(model: Pipeline) -> dict[str, float]:
    predictions = model.predict(X_test)
    fraud_probabilities = model.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, fraud_probabilities),
    }

def classifier_params(search_params: dict) -> dict:
    return {key.removeprefix("classifier__"): value for key, value in search_params.items()}

def mlflow_params(params: dict) -> dict:
    return {key: "None" if value is None else value for key, value in params.items()}

def log_tuning_run(
    run_name: str,
    tuning_method: str,
    params: dict,
    metrics: dict[str, float],
    cv_f1: float | None = None,
) -> str:
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({
            "tuning_method": tuning_method,
            "model_type": "RandomForestClassifier",
            **mlflow_params(params),
        })
        mlflow.log_metrics({key.lower(): value for key, value in metrics.items()})
        if cv_f1 is not None:
            mlflow.log_metric("best_cv_f1", cv_f1)
        return run.info.run_id

results: list[dict] = []
run_ids: dict[str, str] = {}

2026/09/21 11:36:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/21 11:36:30 INFO mlflow.store.db.utils: Updating database tables


2026/09/21 11:36:33 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_model_tuning' does not exist. Creating a new experiment.


## Untuned Random Forest

In [4]:
untuned_model = make_random_forest_pipeline(forest_n_jobs=2)
untuned_model.fit(X_train, y_train)
untuned_metrics = evaluate_model(untuned_model)
untuned_params = {
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": 2,
}
run_ids["untuned_random_forest"] = log_tuning_run(
    "untuned_random_forest", "untuned", untuned_params, untuned_metrics
)
results.append({"Method": "Untuned Random Forest", **untuned_metrics, "Best_Params": untuned_params})
display(pd.DataFrame([untuned_metrics]).style.format(precision=4))

,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.9502,0.1250,0.0020,0.0040,0.5025


## Grid Search

In [5]:
grid_parameters = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 12],
    "classifier__min_samples_split": [2],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__max_features": ["sqrt"],
}
grid_search = GridSearchCV(
    estimator=make_random_forest_pipeline(forest_n_jobs=1),
    param_grid=grid_parameters,
    scoring="f1",
    cv=3,
    n_jobs=2,
    refit=True,
)
grid_search.fit(X_train, y_train)
grid_metrics = evaluate_model(grid_search.best_estimator_)
grid_best_params = classifier_params(grid_search.best_params_)
run_ids["grid_search"] = log_tuning_run(
    "grid_search",
    "grid_search",
    grid_best_params,
    grid_metrics,
    grid_search.best_score_,
)
results.append({"Method": "Grid Search Random Forest", **grid_metrics, "Best_Params": grid_best_params})
print(f"Combinations evaluated: {len(grid_search.cv_results_['params'])}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")
print("Best parameters:", grid_best_params)
display(pd.DataFrame([grid_metrics]).style.format(precision=4))

Combinations evaluated: 8
Best CV F1: 0.0496
Best parameters: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}


,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.9040,0.0447,0.0467,0.0456,0.5146


## Random Search

In [6]:
random_parameters = {
    "classifier__n_estimators": [80, 120, 160, 220, 300],
    "classifier__max_depth": [None, 8, 12, 18, 25],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2", 0.7],
}
random_search = RandomizedSearchCV(
    estimator=make_random_forest_pipeline(forest_n_jobs=1),
    param_distributions=random_parameters,
    n_iter=10,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=2,
    refit=True,
)
random_search.fit(X_train, y_train)
random_metrics = evaluate_model(random_search.best_estimator_)
random_best_params = classifier_params(random_search.best_params_)
run_ids["random_search"] = log_tuning_run(
    "random_search",
    "random_search",
    random_best_params,
    random_metrics,
    random_search.best_score_,
)
results.append({"Method": "Random Search Random Forest", **random_metrics, "Best_Params": random_best_params})
print(f"Candidates evaluated: {len(random_search.cv_results_['params'])}")
print(f"Best CV F1: {random_search.best_score_:.4f}")
print("Best parameters:", random_best_params)
display(pd.DataFrame([random_metrics]).style.format(precision=4))

Candidates evaluated: 10
Best CV F1: 0.0858
Best parameters: {'n_estimators': 160, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.7, 'max_depth': 8}


,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.6799,0.0509,0.3124,0.0876,0.5067


## Bayesian Optimization

In [7]:
cross_validation = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=RANDOM_STATE
)

def optuna_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 300, step=20),
        "max_depth": trial.suggest_categorical("max_depth", [None, 8, 12, 18, 25]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.7]),
    }
    candidate = make_random_forest_pipeline(params, forest_n_jobs=1)
    scores = cross_val_score(
        candidate,
        X_train,
        y_train,
        scoring="f1",
        cv=cross_validation,
        n_jobs=2,
    )
    return float(np.mean(scores))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(optuna_objective, n_trials=15, show_progress_bar=False)

optuna_best_params = study.best_params
optuna_model = make_random_forest_pipeline(optuna_best_params, forest_n_jobs=2)
optuna_model.fit(X_train, y_train)
optuna_metrics = evaluate_model(optuna_model)
run_ids["bayesian_optuna"] = log_tuning_run(
    "bayesian_optuna",
    "bayesian_optuna",
    optuna_best_params,
    optuna_metrics,
    study.best_value,
)
results.append({"Method": "Bayesian/Optuna Random Forest", **optuna_metrics, "Best_Params": optuna_best_params})
print(f"Trials completed: {len(study.trials)}")
print(f"Best CV F1: {study.best_value:.4f}")
print("Best parameters:", optuna_best_params)
display(pd.DataFrame([optuna_metrics]).style.format(precision=4))

Trials completed: 15
Best CV F1: 0.0631
Best parameters: {'n_estimators': 100, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 0.7}


,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.8641,0.0460,0.0892,0.0607,0.4995


## Model Comparison

In [8]:
logistic_baseline = {
    "Method": "Logistic Regression Baseline",
    "Accuracy": 0.5023,
    "Precision": 0.0466,
    "Recall": 0.4686,
    "F1": 0.0848,
    "ROC_AUC": 0.4876,
    "Best_Params": {"class_weight": "balanced", "max_iter": 1000, "random_state": 42},
}
comparison = pd.DataFrame([logistic_baseline, *results])[
    ["Method", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "Best_Params"]
]
comparison["Best_Params"] = comparison["Best_Params"].map(
    lambda params: json.dumps(params, sort_keys=True)
)
display(comparison.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "ROC_AUC": "{:.4f}",
}))

,Method,Accuracy,Precision,Recall,F1,ROC_AUC,Best_Params
0,Logistic Regression Baseline,0.5023,0.0466,0.4686,0.0848,0.4876,"{""class_weight"": ""balanced"", ""max_iter"": 1000, ""random_state"": 42}"
1,Untuned Random Forest,0.9502,0.1250,0.0020,0.0040,0.5025,"{""class_weight"": ""balanced"", ""n_jobs"": 2, ""random_state"": 42}"
2,Grid Search Random Forest,0.9040,0.0447,0.0467,0.0456,0.5146,"{""max_depth"": 12, ""max_features"": ""sqrt"", ""min_samples_leaf"": 2, ""min_samples_split"": 2, ""n_estimators"": 100}"
3,Random Search Random Forest,0.6799,0.0509,0.3124,0.0876,0.5067,"{""max_depth"": 8, ""max_features"": 0.7, ""min_samples_leaf"": 4, ""min_samples_split"": 2, ""n_estimators"": 160}"
4,Bayesian/Optuna Random Forest,0.8641,0.0460,0.0892,0.0607,0.4995,"{""max_depth"": 12, ""max_features"": 0.7, ""min_samples_leaf"": 5, ""min_samples_split"": 2, ""n_estimators"": 100}"


## MLflow Runs

In [9]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
logged_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
)
run_columns = [
    "tags.mlflow.runName",
    "params.tuning_method",
    "params.model_type",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1",
    "metrics.roc_auc",
]
display(logged_runs[run_columns].sort_values("tags.mlflow.runName"))
print("Tracking URI:", mlflow.get_tracking_uri())
print("Run IDs:", run_ids)

,tags.mlflow.runName,params.tuning_method,params.model_type,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc
0,bayesian_optuna,bayesian_optuna,RandomForestClassifier,0.864126,0.045977,0.089249,0.060690,0.499508
2,grid_search,grid_search,RandomForestClassifier,0.904030,0.044660,0.046653,0.045635,0.514575
1,random_search,random_search,RandomForestClassifier,0.679868,0.050926,0.312373,0.087575,0.506691
3,untuned_random_forest,untuned,RandomForestClassifier,0.950219,0.125000,0.002028,0.003992,0.502550


Tracking URI: sqlite:///C:/Users/Priya Koma/Desktop/AI_ML_Assessments/ai_fraud_detection_poc/mlruns/mlflow.db
Run IDs: {'untuned_random_forest': '77131ad9be63485fbeff44ed1602262f', 'grid_search': '98de05a4c9614fa9b943f88479f036c2', 'random_search': 'f87d5eeb9e5a46aeb5a9686635ee2834', 'bayesian_optuna': 'a6aac51b5f954034aa10c2a3d1eb67d6'}


## Tuning Conclusions

In [10]:
ranked_by_f1 = comparison.sort_values("F1", ascending=False).reset_index(drop=True)
best_result = ranked_by_f1.iloc[0]
print(f"Highest untouched-test F1: {best_result['Method']} ({best_result['F1']:.4f})")
print(f"Its precision is {best_result['Precision']:.4f}, recall is {best_result['Recall']:.4f}, and ROC-AUC is {best_result['ROC_AUC']:.4f}.")
print("Accuracy is reported but is not used alone because the target is highly imbalanced.")
print("These results compare tuning approaches only; no production model is selected or saved.")
display(ranked_by_f1[["Method", "F1", "Precision", "Recall", "ROC_AUC"]].style.format(precision=4))

Highest untouched-test F1: Random Search Random Forest (0.0876)
Its precision is 0.0509, recall is 0.3124, and ROC-AUC is 0.5067.
Accuracy is reported but is not used alone because the target is highly imbalanced.
These results compare tuning approaches only; no production model is selected or saved.


,Method,F1,Precision,Recall,ROC_AUC
0,Random Search Random Forest,0.0876,0.0509,0.3124,0.5067
1,Logistic Regression Baseline,0.0848,0.0466,0.4686,0.4876
2,Bayesian/Optuna Random Forest,0.0607,0.0460,0.0892,0.4995
3,Grid Search Random Forest,0.0456,0.0447,0.0467,0.5146
4,Untuned Random Forest,0.0040,0.1250,0.0020,0.5025
